In [1]:
# 문서 로딩
import fitz
from langchain_core.documents import Document

file_name = "경제위기 이후 우리 성장은 왜 구조적으로 낮아졌는가.pdf"

doc = fitz.open(file_name)

docs = []
for page_num in range(doc.page_count):
    page = doc.load_page(page_num)
    text = page.get_text("text")
    text = text.replace("\n", " ") 
    docs.append(Document(page_content=text, metadata={"page": page_num + 1}))

# text_head에 문서 앞 2페이지 내용 저장
text_all = ""
text_head = ""

for page in docs:
    if page.metadata["page"] < 3:
        text_head += page.page_content
    text_all += page.page_content + "\n\n"

text_head = Document(page_content=text_head)

In [2]:
# langchain 초기화
from langchain_naver import ClovaXEmbeddings
from langchain_naver import ChatClovaX

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate

import numpy as np
from sklearn.cluster import KMeans

llm = ChatClovaX(
    model="HCX-007",
    thinking={
        "effort": "none" 
    },
    max_completion_tokens=5000,
    temperature=0.1,
)

embeddings = ClovaXEmbeddings(
    model='bge-m3'
)

In [3]:
# 문서 분할
text_splitter = RecursiveCharacterTextSplitter(separators=["\n\n", "\n", "\t"," "], chunk_size=2000, chunk_overlap=256)

docs = text_splitter.create_documents([text_all])

num_documents = len(docs)

print(num_documents)

21


In [4]:
# 임베딩
vectors = embeddings.embed_documents([x.page_content for x in docs])

In [ ]:
# k-means 클러스터링 수행
# 클러스터 개수 설정
if num_documents > 7:
    num_clusters = 7
else:
    num_clusters = num_documents

print ("num_clusters", num_clusters)

# K-means clustering (random seed값 고정)
kmeans = KMeans(n_clusters=num_clusters, random_state=42).fit(vectors)

print(kmeans.labels_)

num_clusters 7
[2 1 2 2 5 5 2 0 1 6 6 6 3 1 1 1 1 1 1 4 4]


In [6]:
# cetroid 추출
closest_indices = []

for i in range(num_clusters):
    distances = np.linalg.norm(vectors - kmeans.cluster_centers_[i], axis=1)
    closest_index = np.argmin(distances)
    closest_indices.append(closest_index)

selected_indices = sorted(closest_indices)
selected_indices

[0, 4, 7, 9, 12, 18, 19]

In [7]:
# 문서 chunk 별 논쟁사항 추출
from langchain_core.output_parsers import StrOutputParser


map_prompt = """- 한국은행 보도자료의 한 단락이 제공됩니다. 
- 해당 단락은 세 개의 백틱(```)으로 둘러싸여 있습니다.
- 한국의 경제·사회·문화적 특수성을 고려하여 해당 보도자료 공개시 언론이 악의적으로 꼬투리를 잡거나 갈등·분노·책임 공방 구도로 프레이밍할 수 있는 논쟁 지점을 분석하시오.
- 정부에 비판적인 언론사에서 악의적으로 만들어낼 수 있는 논쟁 지점을 분석하시오.
- 이때 정책의 타당성이나 학술적 균형보다는 정부 정책 비판 또는 정부 정책의 잘못으로 인한 손해나 공격의 대상이 될 수 있는 집단을 중심으로 분석하시오.
- 구체적인 수치가 언급되었다면 적극적으로 인용합니다.

```{text}```"""

map_prompt_template = PromptTemplate.from_template(map_prompt)

map_chain = map_prompt_template | llm | StrOutputParser()

In [8]:
# 선택된 문서 chunk 확인
selected_docs = [docs[doc] for doc in selected_indices]

for doc in selected_docs:
    print(doc)

page_content='1 경제위기 이후 우리 성장은 왜 구조적으로 낮아졌는가? : 기업 투자경로를 중심으로 1   ‌  1990년대 이후 우리 경제는 경제위기를 거치면서 성장추세가 이전 수준을 회복하지 못하고 구조적으로 둔화 되었다. 거시 데이터를 통해 분석한 결과, 이는 위기로 인한 부정적 수요충격이 투자의 이력현상(hysteresis)*을  통해 성장의 추세적 둔화로 이어졌던 것으로 추정되었다. * 일시적 충격이 경제변수(예: 실업률, 투자 등)의 장기 경로에 부정적 영향을 미치는 현상 2   ‌  이러한 투자의 이력현상 원인을 규명하고 해결 방안을 모색하기 위해 기업 단위의 미시 데이터를 추가로 활 용하여 실증 분석하였다. 2,200여 개 외감기업을 대상으로 분석한 결과, 금융위기 이후 소수 대기업을 제외 한 대다수 기업에서 투자가 정체 또는 감소하였으며, 이러한 투자부진은 금융제약보다는 수익성 악화와 밀접 한 관련이 있던 것으로 나타났다. 3   ‌  이처럼 위기 이후 기업의 투자 부진이 금융제약보다는 수익성 악화에 기인한다는 점에서, 금융지원만으로 는 이력현상을 완화하기 어려울 수 있다. 근본적으로는 한계기업들이 자연스럽게 퇴출되는 정화 메커니즘 (cleansing effect)이 정상적으로 작동하도록 하는 동시에, 신생기업의 진입이 원활하게 이루어지도록 하여 경제 의 역동성을 증대시켜야 한다. 4   ‌  만약, 정화 메커니즘이 작동하였다면 우리 투자와 성장이 어떻게 되었을까? 실제 퇴출기업의 특징을 기반 으로 퇴출 고위험기업을 식별한 결과, 2014~19년 중 전체 표본의 3.8%에 해당하는 것으로 추정되었다.    BOK 이슈노트 제 2025-33호 BANK OF KOREA 2025년 11월 12일 경제위기 이후 우리 성장은 왜 구조적으로  낮아졌는가? : 기업 투자경로를 중심으로 이종웅 한국은행 조사국 조사총괄팀 차장 T el. 02-759-4165 jw.lee@bok.or.kr 부유신 한국은행 조사국 조사총괄팀 과장 T el. 02

In [10]:
# 선택된 문서 chunk 논쟁사항 추출
summary_list = []

for i, doc in enumerate(selected_docs):
    chunk_summary = map_chain.invoke({"text": doc.page_content})
    
    summary_list.append(chunk_summary)
    
    print (f"#{i} (chunk #{selected_indices[i]}) - {chunk_summary} \n")

#0 (chunk #0) - 보도자료를 보면, 한국 경제가 1990년대 이후 경제위기를 겪으면서 성장 추세가 둔화되고 있다는 내용이 담겨 있습니다. 이에 대한 원인으로는 '투자의 이력현상'이 지적되고 있으며, 이를 해결하기 위해서는 금융 지원뿐만 아니라 한계기업의 자연스러운 퇴출과 신생기업의 진입이 필요하다는 제안이 포함되어 있습니다.

이러한 내용에서 논쟁 지점이 될 수 있는 부분은 다음과 같습니다:

1. **한계기업의 퇴출**: 한계기업의 퇴출을 촉진해야 한다는 주장은 일부에서는 일자리 감소를 우려하며 반대할 수 있습니다. 특히 중소기업이나 자영업자들이 주요 대상이라면, 이들의 생존권을 위협한다는 비판이 제기될 수 있습니다. 

2. **금융지원만으로는 부족하다는 주장**: 금융지원만으로는 충분치 않다는 주장은 정부의 금융지원 정책에 대해 비판적인 시각을 불러일으킬 수 있습니다. 예를 들어, 정부가 그동안 충분한 지원을 하지 않았다는 비난이나 반대로 과도한 금융지원으로 인해 발생한 부작용에 대한 책임을 묻는 목소리가 있을 수 있습니다.

3. **소수 대기업 중심의 성장**: 보고서에는 금융위기 이후 소수 대기업을 제외한 대다수의 기업들이 투자를 정체하거나 감소했다는 내용이 있습니다. 이는 대기업 중심 성장 전략에 대한 비판으로 이어질 수 있으며, 중소기업 및 스타트업의 성장을 저해하는 구조적 문제에 대한 논의로 확장될 가능성이 있습니다.

언론사들은 이와 같은 부분을 강조하면서 현 정부의 경제정책 실패를 부각시킬 수 있습니다. 또한, 특정 집단(예를 들어 중소기업 소유주나 노동자)에게 피해가 돌아갔음을 강조함으로써 사회적 갈등을 유발하려는 시도가 있을 수도 있습니다. 

#1 (chunk #4) - 한국은행의 보도자료 중 일부 내용을 바탕으로 다음과 같이 논쟁 지점을 분석해보겠습니다:

1. **외환위기 이전과 이후의 성장 동인 변화**:
    - 외환위기 이전에는 공급요인이 성장을 주도했으나, 이후에는 수요요인이 주요 동인으로 작용했다는 내용이 포함

In [11]:
# Text Merge
summaries = "\n".join(summary_list)

summaries = Document(page_content=summaries)


In [13]:
# 논쟁 사항 정리 프롬프트
combine_prompt = """- 문서에서 추출한 문서 개요와 주요 내용을 제공받게 됩니다.
- 문서 개요에 차례나 목차가 있으면 순서에 맞추어 정리합니다.
- 사용자가 문서 전체의 내용을 이해 할 수 있도록 일목 요연하게 정리합니다.
- 한국의 경제·사회·문화적 특수성을 고려하여 해당 보도자료 공개시 언론이 악의적으로 꼬투리를 잡거나 갈등·분노·책임 공방 구도로 프레이밍할 수 있는 논쟁 지점을 분석하시오.
- 정부에 비판적인 언론사에서 악의적으로 만들어낼 수 있는 논쟁 지점을 분석하시오.
- 이때 정책의 타당성이나 학술적 균형보다는 정부 정책 비판 또는 정부 정책의 잘못으로 인한 손해나 공격의 대상이 될 수 있는 집단을 중심으로 분석하시오.
- 구체적인 수치가 언급되었다면 적극적으로 인용합니다.

## 문서 제목 : {file_name}
## 문서 개요
{text_head}

## 주요 내용
```{text}```

## 출력 형식
- 문서 제목
- 문서 개요
- 주요 내용
- 논쟁 지점"""


combine_prompt_template = PromptTemplate.from_template(combine_prompt)

reduce_chain = combine_prompt_template | llm | StrOutputParser()

In [14]:
# 논쟁 사항 정리
output = reduce_chain.invoke({
    "text": summaries.page_content,
    "text_head": text_head.page_content,
    "file_name": file_name
})
print (output)

문서 제목: 경제위기 이후 우리 성장은 왜 구조적으로 낮아졌는가?

문서 개요

1990년대 이후 우리 경제는 경제위기를 거치면서 성장추세가 이전 수준을 회복하지 못하고 구조적으로 둔화되었습니다. 거시 데이터를 통해 분석한 결과, 이는 위기로 인한 부정적 수요충격이 투자의 이력현상(hysteresis)을 통해 성장의 추세적 둔화로 이어졌던 것으로 추정되었습니다. 이러한 투자의 이력현상 원인을 규명하고 해결 방안을 모색하기 위해 기업 단위의 미시 데이터를 추가로 활용하여 실증 분석하였습니다. 2,200여 개 외감기업을 대상으로 분석한 결과, 금융위기 이후 소수 대기업을 제외한 대다수 기업에서 투자가 정체 또는 감소하였으며, 이러한 투자부진은 금융제약보다는 수익성 악화와 밀접한 관련이 있던 것으로 나타났습니다. 이처럼 위기 이후 기업의 투자 부진이 금융제약보다는 수익성 악화에 기인한다는 점에서, 금융지원만으로는 이력현상을 완화하기 어려울 수 있습니다. 근본적으로는 한계기업들이 자연스럽게 퇴출되는 정화 메커니즘이 정상적으로 작동하도록 하는 동시에, 신생기업의 진입이 원활하게 이루어지도록 하여 경제의 역동성을 증대시켜야 합니다. 만약, 정화 메커니즘이 작동하였다면 우리 투자와 성장이 어떻게 되었을까? 실제 퇴출기업의 특징을 기반으로 퇴출 고위험기업을 식별한 결과, 2014~19년 중 전체 표본의 3.8%에 해당하는 것으로 추정되었습니다. 반면 실제 퇴출된 기업은 2.0%로 고위험기업의 절반 수준에 그쳐 그간 정화효과가 미흡했던 것으로 분석됩니다. 이러한 퇴출 고위험기업이 실제로 퇴출되고 산업 내 정상기업으로 대체되었다면, 해당 기간 중 국내 투자는 +3.3%, GDP는 +0.5% 증가하였을 것으로 추정됩니다. 팬데믹 이후(2022~24년) 기간을 보면 퇴출 고위험기업 비중(3.8%)은 이전 시기와 유사하였으나 실제 퇴출기업 비중은 0.4%로 더욱 낮아지며, 이들이 성공적으로 대체되었다면 투자는 +2.8%, GDP는 +0.4% 증가할 수 있었을 것으로 추정됩니다. 종합

In [15]:
system_prompt = """당신은 “AI 공보관(Press Risk Forecaster)”입니다.

임무:
보도자료가 엠바고 해제될 경우,
언론이 선택할 가능성이 높은 기사 ‘제목 프레이밍’을
사전 예측하는 내부 리스크 시뮬레이션을 수행합니다.

원칙:
- 실제 기사 작성이 아닌 예측·분석 목적입니다.
- 부정적·자극적 표현은 문체적 효과로만 사용합니다.
- 특정 인물·기관을 비방하지 않습니다.

사실 사용 규칙:
- 모든 수치·발언·정책 내용은 보도자료 본문에서만 추출합니다.
- 본문에 없는 사건, 발언, 외부 평가를 생성하지 않습니다.
- 과장·감성은 표현 방식에만 적용하며 사실은 변경하지 않습니다."""

user_prompt = """다음 보도자료 본문을 바탕으로, 엠바고 해제 시 언론이 작성할 가능성이 높은 기사 제목을 예측하는 문체 시뮬레이션을 수행하십시오.
작업은 아래 Pipeline을 순서대로 따르십시오.

--------------------------------
[1단계: Fact Pool 고정]
보도자료에서 직접 확인 가능한 사실만 추출하십시오.

[Fact Pool]
- 정책/조치
- 공식 수치
- 공식 발언(요지)
- 명시된 평가/판단
- 긍정 요소
- 해석 여지(본문 표현을 언론이 책임·결과·손실 등 부정적으로 확대 해석할 수 있는 지점)

※ 이후 모든 제목은 이 Fact Pool 내 정보만 사용하십시오.
※ 단, ‘해석 여지’ 항목에 포함된 내용은 인과·책임·결과가 암시되도록 제목 수준에서 재배열하는 것은 허용합니다.
※ 본문 자료 외 외부 사건·발언·수치는 생성 금지입니다.

--------------------------------
[2단계: 문체 패턴 적용]
아래 패턴과 단어를 조합하여 제목을 설계하십시오.

[패턴]
a. 수치 재프레이밍: 불리한 수치 강조
- Fact Pool : 공식 수치: 소비자심리지수 97.5 (전월 대비 2.5p 하락)
- 제목 예시 : ""소비심리 100선 붕괴… 2.5p 하락에 시장 ‘불안’ 확산""/ ""기준선 밑으로 내려앉은 소비심리, 하락 폭보다 큰 불안""

b. 수치 손실화: 성장 기회·잠재 규모가 ‘사라진 것처럼’ 표현
- Fact Pool : 공식 수치: 공식 수치: 수출 증가율 1.2%
- 제목 예시 : ""수출 1%대 증가에 그쳐… 성장 기회 ‘사라진’ 산업 현장"", ""1.2% 증가라는 숫자 뒤에 가려진 수출 모멘텀 상실""

c.  발언 대비 충돌: 과거 전망 vs 현재 지표
- Fact Pool : 공식 발언: “경기 회복 흐름이 이어질 것”, 공식 수치: 소비자심리지수 하락
- 제목 예시 : ""“회복 흐름”이라던 전망, 심리지표 하락 앞에 무색""

d. 정책 효과 역전: 의도된 정책 효과가 오히려 역효과를 낳은 듯한 대비
- Fact Pool : 정책/조치: 금리 동결, 공식 수치: 대출금리 상승
- 제목 예시 : ""안정 노린 동결 정책, 체감 금리는 되레 상승""

e. 구조 확장: 일부 변동 → 구조 문제인 것처럼 확대 해석
- Fact Pool : 공식 수치: 특정 항목 가격 상승
- 제목 예시 : ""일부 품목 상승이 전반적 물가 불안으로 번지나"", ""가격 변동 넘어 구조적 물가 압박 신호?""

f. 책임 암시 확장: 구조 문제를 ‘방치·지연의 결과’처럼 서술
- Fact Pool : 명시된 평가: “불확실성 지속”, 정책/조치: 추가 대응 언급 없음
- 제목 예시 : ""불확실성 지속 속 대응은 제자리… 결국 부담은 시장 몫"", ""구조적 불안 누적, 선택의 지연이 낳은 결과인가""

g. 감정 과장: 정서적 단어 전면 배치
- Fact Pool : 공식 수치: 소비자심리지수 하락
- 제목 예시 : ""소비심리 하락에 커지는 경기 불안"", ""숫자보다 무거운 체감… 시장에 번지는 공포""

[단어]
- 위기 단어: 급감, 추락, 불안, 하락, 제동, 위기, 공포, 경착륙 등
- 결과 단어: 발목, 잠식, 갉아먹다, 묶이다, 사라지다, 놓치다
- 반전 연결어: 하지만, 그러나, 반면
- 대비 강조: 무색, 엇박자, 정반대
- 귀결 표현: 결국, 되레, 오히려, 결과적으로

--------------------------------
[3단계: 감성 적용]
2단계의 문체 패턴에 다음 감성을 조합하시오.
- 위기 / 충돌 / 축소 / 불안/ 상실

--------------------------------

[출력 요구사항]
보도자료 성격에 따라 헤드라인 프레이밍 카테고리를 먼저 도출 (카테고리는 고정 아님)
1) 헤드라인 카테고리명, 예상작성 빈도, 제목만 출력. 수식어·부연 설명·분석 문장 출력 금지
2) 헤드라인 카테고리별 제목 2~3개 제시
3) 카테고리별 예상 작성 빈도 표시 (높음 / 중간 / 낮음)
4) 보도자료에 직접 언급된 경우에만  총재 발언 또는 최근 경제 이슈를 연결 (없으면 ‘연결 없음’)
5) 각 제목은 ‘사실 서술’이 아니라 ‘언론이 일반 시민들을 선동할 수 있는, 책임·손실·구조 문제를 연상시키도록 의도적으로 재배열한 결과물’이어야 합니다.

--------------------------------

[보도자료 본문]
{content}"""

In [ ]:
# 문서 요약을 사용한 헤드라인 추출
from langchain_core.messages import SystemMessage, HumanMessage

user_message = user_prompt.format(content=output)

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_message)
]

response = llm.invoke(messages)
print(response.content)


**헤드라인 카테고리명**: 경제성장 둔화 원인 분석  
**예상 작성 빈도**: 높음  

1. **금융지원만으로 부족한 경제 회복**  
   - "경제위기 후 성장 둔화, 금융지원만으로는 해결 어렵다"  
   - "수익성 악화 탓... 금융지원만으로는 경제 회복 난망"  

2. **한계기업 퇴출 필요성 대두**  
   - "한계기업 방치, 경제 정화 메커니즘 마비시켰다"  
   - "퇴출되지 않은 한계기업, 경제 성장의 발목 잡아"  

3. **신생기업 진입 장벽 해소 촉구**  
   - "신생기업 진입 막힌 한국 경제, 역동성 잃었다"  
   - "새로운 기업 진입 없이는 성장 반등 어려워"  

**헤드라인 카테고리명**: 경제정책 비판  
**예상 작성 빈도**: 중간  

1. **정부 정책 실패 지적**  
   - "정부의 경제정책, 성장 둔화 못 막았다"  
   - "경제위기 이후 정부 대응, 성장 추세 둔화 초래했나"  

2. **소수 대기업 중심 성장 논란**  
   - "대기업 중심 성장, 중소기업 희생 강요했다"  
   - "대기업 편향적 정책이 경제 활력 떨어뜨렸다"  

**헤드라인 카테고리명**: 경제 구조적 문제 진단  
**예상 작성 빈도**: 중간  

1. **투자의 이력현상 문제점 조명**  
   - "경제위기 충격, 장기적 투자 침체로 이어졌다"  
   - "투자 이력현상이 가져온 성장 둔화의 늪"  

2. **경제 정화 메커니즘 작동 부재**  
   - "경제 정화 기능 약화, 성장 잠재력 갉아먹어"  
   - "경제 정화 메커니즘 고장으로 성장 둔화 가속화"  

**헤드라인 카테고리명**: 경제 회복 가능성 탐색  
**예상 작성 빈도**: 낮음  

1. **미래 성장 동력 확보 방안 제시**  
   - "혁신적 신산업 투자 촉진이 답이다"  
   - "규제 완화로 새 성장 동력 찾아야 한다"  

2. **GDP 성장 전망 긍정적 시나리오**  
   - "정상적 기업 교체 있었